# Day 2 Project — Engineering Knowledge Assistant

Everything from notebooks 01-07 in one object: chunking, embeddings, retrieval, grounded
generation with citation validation, visible state, and two separate evaluations.

```text
documents -> chunks -> embeddings -> index
question  -> retrieve -> evidence context -> answer + citations -> validation -> state
```

## Before you begin

### Learning outcomes

- Assemble the reference project and read its state end to end.
- Produce a scorecard that separates retrieval failures from answer failures, then fix one.

Architecture reference: [D06–D07](../../diagrams/source/day_02.md).

### Expected observation

A ten-case report where two retrieval misses are visible by id, and where swapping only
the embedder repairs both without touching the generator.

### Modes

The evaluation always runs on the offline generator so a full report costs nothing. If a
key is present, exactly one live answer is generated for comparison.

## Concept briefing

## What to carry into Day 3

Knowledge usually comes from an external corpus. Memory usually records selected
information from interactions. Neither should be confused with active context. Day 3
shows how history grows, why summaries lose information, and how persistent memory and
execution policy require explicit lifecycle controls.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) The reference project lives in run_project.py next to this day's src/.
sys.path.insert(0, str(PROJECT_ROOT))

from knowledge_agent.evaluation import (
    evaluate_answers,
    evaluate_retrieval,
    load_golden_set,
    render_table,
    summarize,
    summarize_detail,
    summarize_essential_terms,
)
from run_project import build_assistant

cases = load_golden_set(PROJECT_ROOT / "data" / "golden_set.json")
print("Golden cases:", len(cases))

## Step 1 — Build the assistant

`build_assistant("mock")` wires the deterministic parts: hash embedder, in-memory index,
offline generator. Same objects you built by hand in notebooks 01-05, assembled once.

In [ ]:
assistant = build_assistant("mock")

print("index      :", type(assistant.index).__name__)
print("embedder   :", type(assistant.index.embedder).__name__)
print("generator  :", type(assistant.generator).__name__)
print("top_k      :", assistant.top_k)
print("chunks     :", len(assistant.index.chunks))

## Step 2 — One question, all the way through

`KnowledgeAssistant.answer` retrieves, generates, validates the citations and records
every intermediate result in a `KnowledgeState`.

In [ ]:
state = assistant.answer("Does requesting island mode immediately open the grid breaker?")

print("status    :", state.status)
print("error     :", state.error)
print()
print("retrieved :")
for item in state.retrieved:
    print(f"   rank {item.rank}  score {item.score:.3f}  {item.chunk.chunk_id:42} {item.chunk.section}")
print()
print("abstained :", state.answer.abstained)
print("grounded  :", state.answer.grounded)
print("citations :", [citation.chunk_id for citation in state.answer.citations])
print("answer    :", state.answer.answer[:220], "...")

## Step 3 — Optional: one live answer

If a key is present we generate the same answer once with the real model and compare. One
call, not ten: the evaluation below stays offline on purpose.

In [ ]:
if not LIVE:
    print("MOCK mode: skipping the live call. Add OPENROUTER_API_KEY to .env to try it.")
else:
    from knowledge_agent.generation import OpenRouterGroundedGenerator
    from knowledge_agent.assistant import KnowledgeAssistant

    try:
        live_assistant = KnowledgeAssistant(assistant.index, OpenRouterGroundedGenerator(), top_k=3)
        live_state = live_assistant.answer("Does requesting island mode immediately open the grid breaker?")
        print("status    :", live_state.status, live_state.error or "")
        if live_state.answer:
            print("abstained :", live_state.answer.abstained)
            print("grounded  :", live_state.answer.grounded)
            print("citations :", [citation.chunk_id for citation in live_state.answer.citations])
            print("dropped   :", [citation.chunk_id for citation in live_state.answer.dropped_citations])
            print("answer    :", live_state.answer.answer)
    except Exception as exc:
        print("Live call failed, the offline result above still stands:", exc)

## Step 4 — The ten-case retrieval report

Retrieval first, on its own. `n/a` marks the unanswerable case, which has no expected
evidence and therefore cannot pass or fail this check.

In [ ]:
retrieval = evaluate_retrieval(assistant.index, cases, top_k=3)
print(render_table(retrieval, ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print()
print("rates :", summarize(retrieval, ["source_hit", "section_hit"]))
print("counts:", summarize_detail(retrieval, ["source_hit", "section_hit"]))

## Step 5 — The ten-case answer report

Different questions: did it abstain when it should, did it cite the expected source, did
every citation survive validation, and did the text contain the essential facts.

In [ ]:
answers = evaluate_answers(assistant, cases)
print(render_table(answers, ["id", "answerable", "abstained", "abstention_correct",
                             "citation_correct", "citation_provenance_ok", "essential_term_coverage"]))
print()
fields = ["completed", "abstention_correct", "citation_correct", "citation_provenance_ok"]
print("rates :", summarize(answers, fields))
print("counts:", summarize_detail(answers, fields))
print("terms :", summarize_essential_terms(answers))

## Step 6 — Diagnose before changing anything

Name each failure and the layer it belongs to. The worked diagnosis for case q01 is in
[`reference/rag_failure_diagnosis.md`](../../reference/rag_failure_diagnosis.md).

In [ ]:
by_id = {case.id: case for case in cases}
for record in retrieval:
    if record["answerable"] and not record["section_hit"]:
        case = by_id[record["id"]]
        print(record["id"], "-", case.question)
        print("   expected :", case.expected_source, "|", case.expected_section)
        print("   retrieved:", record["retrieved_ids"])
        print("   layer    : representation/retrieval - the question paraphrases the document")
print()
for record in answers:
    if not record["abstention_correct"]:
        print(record["id"], "abstained on an answerable question")
        print("   layer    : downstream of retrieval - the section was never supplied")

## Step 7 — Change one layer and re-measure

Everything above used the offline hash embedder. `build_assistant("classroom")` swaps in a
trained embedder (and Chroma if installed) and prints what it could not load. The
generator is left offline so this stays free.

In [ ]:
classroom = build_assistant("classroom", verbose=True)
print("embedder now:", type(classroom.index.embedder).__name__)
print("generator   :", type(classroom.generator).__name__)
print()

retrieval_after = evaluate_retrieval(classroom.index, cases, top_k=3)
print("before:", summarize(retrieval, ["source_hit", "section_hit"]))
print("after :", summarize(retrieval_after, ["source_hit", "section_hit"]))
print()
for before, after in zip(retrieval, retrieval_after):
    if before["section_hit"] != after["section_hit"]:
        print(f"{before['id']}: section_hit {before['section_hit']} -> {after['section_hit']}"
              f" (expected chunk rank {before['expected_rank']} -> {after['expected_rank']})")

## Step 8 — Confirm the answers moved too

A retrieval fix is only real if the answer report agrees. Re-run the answer evaluation on
the improved index, still with the offline generator, and compare the two scorecards.

In [ ]:
from knowledge_agent.assistant import KnowledgeAssistant
from knowledge_agent.generation import MockGroundedGenerator

improved = KnowledgeAssistant(classroom.index, MockGroundedGenerator(), top_k=3)
answers_after = evaluate_answers(improved, cases)

print("before:", summarize(answers, fields))
print("after :", summarize(answers_after, fields))
print("terms before:", summarize_essential_terms(answers))
print("terms after :", summarize_essential_terms(answers_after))
print()
print(render_table(answers_after, ["id", "abstained", "abstention_correct", "citation_correct",
                                   "essential_term_coverage"]))
print()
for record in answers_after:
    if record["essential_terms_total"] and record["essential_term_coverage"] == 0:
        print(record["id"], "still covers none of its essential terms although retrieval now")
        print("   supplies the right section. Look at the rank: the expected chunk is rank 3,")
        print("   and the offline generator quotes whichever retrieved chunk matches the most")
        print("   question words. That is a GENERATION limit now, not a retrieval one -")
        print("   exactly the kind of hand-over the two separate reports are built to show.")

### Try it yourself

Add one question of your own to the golden set *in memory* (not on disk) and score it.
Predict whether the assistant abstains before you run the cell.

In [ ]:
# --- Worked solution ---
from knowledge_agent.schemas import GoldenCase

my_cases = [
    GoldenCase(
        id="mine-01",
        question="What is recorded during the monthly visual inspection?",
        answerable=True,
        expected_source="battery_safety.md",
        expected_section="Inspection",
        essential_terms=["corrosion", "cable damage"],
    ),
    GoldenCase(
        id="mine-02",
        question="Which supplier services the inverters?",
        answerable=False,
        expected_source=None,
        expected_section=None,
        essential_terms=[],
    ),
]
print(render_table(evaluate_retrieval(improved.index, my_cases, top_k=3),
                   ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print()
print(render_table(evaluate_answers(improved, my_cases),
                   ["id", "abstained", "abstention_correct", "citation_correct", "essential_term_coverage"]))
print()
from knowledge_agent.generation import distinctive_matches
for case in my_cases:
    if not case.answerable:
        evidence = improved.index.search(case.question, top_k=3)
        matched = {chunk_id: words for chunk_id, words in distinctive_matches(case.question, evidence).items() if words}
        print(case.id, "matched words:", matched)
        print("   The corpus never names a supplier, but the word 'inverter' appears, so the")
        print("   offline generator believes it has evidence and answers instead of abstaining.")
        print("   A live model reads the passage and abstains - run this again with a key.")
print()
print("A golden case is just a question plus what you already know about the answer.")
print("Ten of them, kept honest, are worth more than any single impressive demo.")

### Checkpoint

**1. Your assistant answers a question wrongly. What do you inspect first, and why?**

<details><summary>Show answer</summary>

The retrieved chunks and their scores - before touching the prompt. If the expected
section is not in the list, no wording change can help and the work belongs in chunking,
the embedder or top-k. Only when the right evidence *was* supplied does the defect move to
context assembly, instructions, generation or citation validation.

</details>

**2. What is the difference between knowledge, context and state in this project?**

<details><summary>Show answer</summary>

Knowledge is the indexed corpus on disk - large, reusable, not in any request. Context is
the few passages plus instructions we put into one model call, and it disappears afterwards.
State is the `KnowledgeState` object the application owns during the run: question,
retrieved chunks, answer, status, error. Day 3 adds the fourth thing - memory, which is
what deliberately survives between runs.

</details>

### Recap

- **Limitation we saw:** an assistant that only prints a final answer hides which layer
  failed and how much of the truth it actually contained.
- **Layer we added:** the assembled project - retrieval, grounded generation, citation
  validation, visible state - plus two separate scorecards over ten known cases.
- **Evidence it worked:** the same golden set names two retrieval misses, and swapping
  only the embedder repairs both, with the answer report moving in step.

Day 2 gave the agent knowledge it can quote. Day 3 handles what happens when the
conversation grows, what deserves to be remembered, and which actions need permission.